In [1]:
import pandas as pd
import os

# Load the rationales data
data_path = "../../crest/data/rationales/e_snli_test_sparsemap_30p.tsv"
df = pd.read_csv(data_path, sep='\t')

In [3]:
import numpy as np
import ast

def process_text_and_rationales(text, z_values):
    """
    Convert tokenized text to normal form and extract rationales only from the first sentence (premise).
    
    Args:
        text (str): Tokenized text with ▁ markers, containing premise and hypothesis separated by </s>
        z_values (list): Binary rationale values for each token
    
    Returns:
        tuple: (normal_text, word_rationales, rationale_ranges_first_sentence)
    """
    # Split text into tokens
    tokens = text.split()
    
    # Parse z_values if it's a string representation of a list
    if isinstance(z_values, str):
        z_values = ast.literal_eval(z_values)
    
    # Find the separator </s> to split premise and hypothesis
    separator_indices = [i for i, token in enumerate(tokens) if token == '</s>']
    
    if len(separator_indices) < 1:
        # If no separator found, treat entire text as one sentence
        sentence_tokens = tokens
        sentence_z_values = z_values
    else:
        # Extract first sentence (premise) - before first </s>
        first_sep_idx = separator_indices[0]
        sentence_tokens = tokens[:first_sep_idx]
        sentence_z_values = z_values[:first_sep_idx]
    
    # Process all tokens to get normal text (for display purposes)
    all_words = []
    current_word = ""
    
    for token in tokens:
        if token == '</s>':
            if current_word:
                all_words.append(current_word)
                current_word = ""
            continue
            
        if token.startswith('▁'):
            if current_word:
                all_words.append(current_word)
            current_word = token[1:]  # Remove ▁
        else:
            current_word += token
    
    if current_word:
        all_words.append(current_word)
    
    normal_text = ' '.join(all_words)
    
    # Process only the first sentence for rationales
    words_first_sentence = []
    word_rationales_first = []
    current_word = ""
    current_word_has_rationale = False
    
    for i, token in enumerate(sentence_tokens):
        z_val = sentence_z_values[i] if i < len(sentence_z_values) else 0.0
        
        if token.startswith('▁'):
            # Start of a new word
            if current_word:
                # Save the previous word
                words_first_sentence.append(current_word)
                word_rationales_first.append(1 if current_word_has_rationale else 0)
            
            # Start new word (remove ▁ marker)
            current_word = token[1:]  # Remove ▁
            current_word_has_rationale = (z_val == 1.0)
        else:
            # Continuation of current word
            current_word += token
            if z_val == 1.0:
                current_word_has_rationale = True
    
    # Don't forget the last word
    if current_word:
        words_first_sentence.append(current_word)
        word_rationales_first.append(1 if current_word_has_rationale else 0)
    
    return normal_text, word_rationales_first

# Apply the processing to all rows
df['normal_text'] = ''
df['word_rationales'] = ''

for idx, row in df.iterrows():
    normal_text, word_rationales = process_text_and_rationales(row['texts'], row['z'])
    df.at[idx, 'normal_text'] = normal_text
    df.at[idx, 'word_rationales'] = word_rationales

print("Text processing completed!")
print(f"Sample processed text:\n{df.iloc[0]['normal_text']}")
print(f"Original tokens: {len(df.iloc[0]['texts'].split())}")
print(f"Words after processing: {len(df.iloc[0]['normal_text'].split())}")
print(f"Premise word rationales: {df.iloc[0]['word_rationales']}")

# Let's also examine the structure to verify it's working correctly
print("\nDetailed breakdown of first example:")
first_text = df.iloc[0]['texts']
print(f"Full text: {first_text}")
sentences = first_text.split(' </s> ')
print(f"First sentence (premise): {sentences[0]}")
if len(sentences) > 1:
    print(f"Second sentence (hypothesis): {sentences[1].replace(' </s>', '')}")

Text processing completed!
Sample processed text:
A man in green pants and blue shirt pushing a cart. A woman is smoking a cigarette.
Original tokens: 24
Words after processing: 17
Premise word rationales: [0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1]

Detailed breakdown of first example:
Full text: ▁A ▁man ▁in ▁green ▁pants ▁and ▁blue ▁shirt ▁pushing ▁ a ▁cart . </s> ▁A ▁woman ▁is ▁smoking ▁ a ▁ cigarette . </s>
First sentence (premise): ▁A ▁man ▁in ▁green ▁pants ▁and ▁blue ▁shirt ▁pushing ▁ a ▁cart .
Second sentence (hypothesis): ▁A ▁woman ▁is ▁smoking ▁ a ▁ cigarette .


In [4]:
# Save the processed DataFrame with rationales
save_path = "processed_e_snli_rationales.csv"
df.to_csv(save_path, index=False)
print(f"Saved processed DataFrame to {save_path}")
print(f"DataFrame shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Saved processed DataFrame to processed_e_snli_rationales.csv
DataFrame shape: (6598, 6)
Columns: ['texts', 'labels', 'predictions', 'z', 'normal_text', 'word_rationales']
